In [1]:
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_core.documents import Document

## Step 1: Sample documents

In [4]:
docs = [
    Document(page_content="LangChain helps build LLM applications."),
    Document(page_content="Pinecone is a vector database for semantic search."),
    Document(page_content="The Eiffel Tower is located in Paris."),
    Document(page_content="Langchain can be used to develop agentic ai application."),
    Document(page_content="Langchain has many types of retrievers.")
]

## Step 2: Dense Retriever (FAISS + Ollama)

In [5]:
embedding_model = OllamaEmbeddings(
    model = "qwen3-embedding:8b"
)

dense_vectorstore = FAISS.from_documents(docs, embedding_model)
dense_retriever = dense_vectorstore.as_retriever()

## Step 3: Sparse Retriever(BM25)

In [6]:
sparse_retriever =  BM25Retriever.from_documents(docs)
sparse_retriever.k = 3 # top- k documents to retriever

## Step 4: Combine with Ensemble Retriever

In [7]:
hybrid_retriever = EnsembleRetriever(
    retrievers = [dense_retriever, sparse_retriever],
    weight = [0.7, 0.3]
)

In [8]:
hybrid_retriever

EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000020E9C0FF0E0>, search_kwargs={}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x0000020E9C0FDD30>, k=3)], weights=[0.5, 0.5])

## Step 5: Query and get results

In [9]:
query = "How can I build an application using LLMs?"
results = hybrid_retriever.invoke(query)

## Step 6: Print results

In [10]:
for i, doc in enumerate(results):
    print(f"\n🔹 Document {i+1}:\n{doc.page_content}")


🔹 Document 1:
LangChain helps build LLM applications.

🔹 Document 2:
Langchain can be used to develop agentic ai application.

🔹 Document 3:
Langchain has many types of retrievers.

🔹 Document 4:
Pinecone is a vector database for semantic search.


## RAG Pipeline with hybrid retriever

In [12]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain

## Step 7: Prompt Template

In [13]:
prompt = PromptTemplate.from_template("""
Answer the question based on the context below.

Context:
{context}

Question: {input}
""")

## Step 8: LLM

In [18]:
llm = ChatOllama(model="deepseek-r1:14b",temperature=0.2)
llm

ChatOllama(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14'}}, output_version=None, model='deepseek-r1:14b', temperature=0.2)

### Create stuff Docuemnt Chain

In [19]:
document_chain = create_stuff_documents_chain(llm = llm, prompt = prompt)

## Create Full RAG chain

In [20]:
rag_chain = create_retrieval_chain(
    retriever = hybrid_retriever,
    combine_docs_chain = document_chain
)
rag_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000020E9C0FF0E0>, search_kwargs={}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x0000020E9C0FDD30>, k=3)], weights=[0.5, 0.5]), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nAnswer the question based on the context below.\n\nContext:\n{context}\n\nQuestion: {input}\n')
            | ChatOllama(metadata={'l

In [21]:
# Step 9: Ask a question
query = {"input": "How can I build an app using LLMs?"}
response = rag_chain.invoke(query)

# Step 10: Output
print("✅ Answer:\n", response["answer"])

print("\n📄 Source Documents:")
for i, doc in enumerate(response["context"]):
    print(f"\nDoc {i+1}: {doc.page_content}")

✅ Answer:
 

To build an app using LLMs, you can utilize LangChain, a framework designed for developing applications with large language models. Here's a structured approach:

1. **Framework Setup**: Use LangChain to set up your application. LangChain provides tools for integrating LLMs and building agentic AI applications, which can make autonomous decisions and take actions.

2. **Integration with Vector Database**: Connect LangChain with Pinecone, a vector database optimized for semantic search. This integration allows your app to efficiently retrieve relevant information from structured data sources.

3. **Agentic AI Implementation**: Implement an agent within your application. This agent will analyze user inputs, decide whether to retrieve information from Pinecone, and then use the LLM to process that information to generate tailored responses or actions.

4. **Dynamic Processing**: The app will dynamically access information using Pinecone and then leverage the LLM to create con